In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision.transforms as transforms
import torchvision.datasets as datasets
from torch.utils.data import DataLoader
import random
import numpy as np
from tqdm import tqdm
from collections import deque

# Set seed for reproducibility
torch.manual_seed(42)
np.random.seed(42)
random.seed(42)

# Set device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

augmentation_pool = [
    transforms.RandomRotation(15),  # Randomly rotate the image within 15 degrees
    transforms.RandomHorizontalFlip(),  # Flip the image horizontally with a 50% chance
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),  # Adjust brightness, contrast, and saturation
    # transforms.RandomGrayscale(p=0.2),  # Randomly convert to grayscale with 20% chance
    transforms.RandomAffine(degrees=0, translate=(0.1, 0.1)),  # Apply small random translations (10% of the image size)
    transforms.RandomPerspective(distortion_scale=0.2, p=0.5),  # Apply perspective transformations
    # transforms.RandomErasing(p=0.5, scale=(0.02, 0.2), ratio=(0.3, 3.3), value="random"),  # Randomly erase patches
    transforms.RandomCrop(32, padding=4),  # Crop with padding, ensuring the final size is 32x32
    transforms.GaussianBlur(kernel_size=(3, 3), sigma=(0.1, 2.0)),  # Apply Gaussian blur with random sigma
    # transforms.RandomInvert(p=0.2),  # Invert colors with a 20% chance
    # transforms.AutoAugment(transforms.AutoAugmentPolicy.CIFAR10)  # AutoAugment with CIFAR10 policy
]

In [ ]:
# Dataset and DataLoader
transform = transforms.Compose([transforms.ToTensor()])
train_dataset = datasets.CIFAR10(root='./data', train=True, download=True, transform=transform)
test_dataset = datasets.CIFAR10(root='./data', train=False, download=True, transform=transform)

In [ ]:
train_dataset.data = train_dataset.data[:4096]
train_dataset.targets = train_dataset.targets[:4096]

test_dataset.data = test_dataset.data[:1024]
test_dataset.targets = test_dataset.targets[:1024]

In [ ]:
import numpy as np
from torchvision.transforms.functional import to_pil_image, to_tensor

def augment_dataset(image):
    # augmented_data = []
    # augmented_targets = []
    # for i in range(num_augmented):
    #     idx = np.random.randint(0, len(dataset.data))
    #     image = dataset.data[idx]
    #     label = dataset.targets[idx]
    #     augmentation = random.choice(augmentation_pool)
    #     augmented_image = augmentation(image)
    #     augmented_data.append(np.array(augmented_image))
    #     augmented_targets.append(label)
    
    # # Concatenate original and augmented data
    # dataset.data = np.concatenate((dataset.data, np.array(augmented_data)))
    # dataset.targets = np.concatenate((dataset.targets, np.array(augmented_targets)))
    augmentation = augmentation_pool[0]
    return augmentation(image)

test_loader_temp = DataLoader(test_dataset, batch_size=1024, shuffle=False)
for images, labels in test_loader_temp:
    images, labels = images.to(device), labels.to(device)
    augmented_images = torch.stack([augment_dataset(img) for img in images])
    new_targets = labels
    augment_dataset = torch.utils.data.TensorDataset(augmented_images, new_targets)
    test_dataset = torch.utils.data.TensorDataset(images, labels)
    break

from torch.utils.data import ConcatDataset

# Assuming dataset1 and dataset2 are instances of torchvision.datasets or custom datasets
combined_dataset = ConcatDataset([augment_dataset, test_dataset])

# Create DataLoaders
train_loader = DataLoader(train_dataset, batch_size=128, shuffle=True)
test_loader = DataLoader(combined_dataset, batch_size=128, shuffle=False)

In [ ]:
augmentation_pool[5](to_pil_image(images[0].cpu()))

In [ ]:
class CNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.network = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.Conv2d(32, 64, kernel_size=3, stride=1, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2, 2), # output: 64 x 16 x 16

            nn.Conv2d(64, 128, kernel_size=3, stride=1, padding=1),
            nn.ReLU(),
            nn.Conv2d(128, 128, kernel_size=3, stride=1, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2, 2), # output: 128 x 8 x 8

            nn.Conv2d(128, 256, kernel_size=3, stride=1, padding=1),
            nn.ReLU(),
            nn.Conv2d(256, 256, kernel_size=3, stride=1, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2, 2), # output: 256 x 4 x 4

            nn.Flatten(), 
            nn.Linear(256*4*4, 1024),
            nn.ReLU(),
            nn.Linear(1024, 512),
            nn.ReLU(),
            nn.Linear(512, 10))
        
    def forward(self, xb):
        return self.network(xb)

In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader
from torchvision.transforms import ToPILImage, ToTensor
import numpy as np

def new_train_data_as_dataset(train_data, augmentation_set):
    augmented_images = []
    augmented_targets = []
    num_splits = len(augmentation_set)
    split_size = int(len(train_data.data) / num_splits)
    print(augmentation_set)

    for i in range(num_splits):
        # Split the data and targets
        start_idx = i * split_size
        end_idx = (i + 1) * split_size if i != num_splits - 1 else len(train_data.data)
        split_data = train_data.data[start_idx:end_idx]
        split_targets = train_data.targets[start_idx:end_idx]
        # Apply the augmentation to the split data
        augmentation = augmentation_set[i]
        for img in split_data:
            img = ToPILImage()(img)  # Convert numpy image to PIL
            augmented_img = augmentation(img)  # Apply augmentation
            augmented_images.append(ToTensor()(augmented_img))  # Convert back to tensor and append

        augmented_targets.extend(split_targets)
    augmented_images = torch.stack(augmented_images)
    augmented_targets = torch.tensor(augmented_targets)
    augmented_dataset = torch.utils.data.TensorDataset(augmented_images, augmented_targets)
    return augmented_dataset

def convert_to_binarylist(n):
    l = []
    while n > 0:
        l.append(n % 2)
        n = n // 2
    
    while len(l) < 7:
        l.append(0)
    l.reverse()
    return l
        

def train_cnn(cnn_model, train_dataset, criterion, optimizer, aug_num):
    cnn_model.train()
    l = convert_to_binarylist(aug_num)
    selected_aug = [augmentation_pool[i] for i in range(7) if l[i] == 1]
    data_aug  = new_train_data_as_dataset(train_dataset, selected_aug)
    loader = DataLoader(data_aug, batch_size=128, shuffle=True)
    total_loss, total_correct = 0, 0
    for epoch in range(100):
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            optimizer.zero_grad()
            outputs = cnn_model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            total_loss += loss.item()
            total_correct += (outputs.argmax(dim=1) == labels).sum().item()
    return total_loss / len(loader), total_correct / len(loader.dataset)

def validate_cnn(cnn_model, loader):
    cnn_model.eval()
    total_correct = 0
    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            outputs = cnn_model(images)
            total_correct += (outputs.argmax(dim=1) == labels).sum().item()
    return total_correct / len(loader.dataset)

In [ ]:
# x = new_train_data_as_dataset(train_dataset, augmentation_pool)
import json
for i in range(1,129):
    l = convert_to_binarylist(i)
    selected_aug = [augmentation_pool[i] for i in range(7) if l[i] == 1]
    model = CNN().to(device)
    loss = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=0.001)
    train_loss, train_acc = train_cnn(model, train_dataset, loss, optimizer, i)
    test_acc = validate_cnn(model, test_loader)
    with open("results.json", "a") as f:
        json.dump({"list":l, "train_loss": train_loss, "train_acc": train_acc, "test_acc": test_acc}, f, indent=4)
    print(f"Train loss: {train_loss:.4f}, Train accuracy: {train_acc:.4f}, Test accuracy: {test_acc:.4f}")
    